In [ ]:
import os

from __future__ import print_function
import cv2
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow.keras import backend
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint
from sklearn.utils import class_weight
import pickle as pkl
from keras.layers import Dense, Dropout, Embedding, LSTM, Bidirectional

from tensorflow.keras.preprocessing.image import ImageDataGenerator
#import keras.backend as K
from sklearn.utils import shuffle
#from skimage.transform import resize
#from skimage.io import imsave
import numpy as np
#from keras.models import Model
from tensorflow.keras.layers import Input, concatenate, Conv2D, MaxPooling2D, AveragePooling2D, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import re
_nsre = re.compile('([0-9]+)')
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(_nsre, s)]

from sklearn.model_selection import train_test_split
import tensorflow as tf
import tensorflow.keras.optimizers
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.models import model_from_json

from tensorflow.keras.layers import Input, Conv2D, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.layers import BatchNormalization
#from tensorflow.keras.utils import multi_gpu_model
from sklearn.model_selection import KFold, StratifiedKFold
from skimage import exposure

from segmentation_models import Unet
from segmentation_models import get_preprocessing
from segmentation_models.losses import bce_jaccard_loss
from segmentation_models.metrics import iou_score

from keras.preprocessing import sequence
from keras.models import Sequential

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
x_data = np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/data256.npy')
y_data = np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/mask_corrected/mask256.npy')

In [ ]:
x_test1=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testdata1/data256.npy')
y_test1=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testmask1/mask256.npy')
x_test2=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testdata2/data256.npy')
y_test2=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testmask2/mask256.npy')
x_test3=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testdata3/data256.npy')
y_test3=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testmask3/mask256.npy')
x_test4=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testdata4/data256.npy')
y_test4=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testmask4/mask256.npy')
x_test5=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testdata5/data256.npy')
y_test5=np.load('/content/drive/MyDrive/Colab Notebooks/airway_truncated_data/train/new_data_trans/test_data/testmask5/mask256.npy')

In [ ]:
x_data= x_data/x_data.max()
y_data= y_data/y_data.max()
x_data = x_data[:,:,:,np.newaxis]
y_data = y_data[:,:,:,np.newaxis]

In [ ]:
x_test1= x_test1/x_test1.max()
y_test1= y_test1/y_test1.max()
x_test1 = x_test1[:,:,:,np.newaxis]
y_test1 = y_test1[:,:,:,np.newaxis]
x_test2= x_test2/x_test2.max()
y_test2= y_test2/y_test2.max()
x_test2 = x_test2[:,:,:,np.newaxis]
y_test2 = y_test2[:,:,:,np.newaxis]

In [ ]:
x_test3= x_test3/x_test3.max()
y_test3= y_test3/y_test3.max()
x_test3 = x_test3[:,:,:,np.newaxis]
y_test3 = y_test3[:,:,:,np.newaxis]
x_test4= x_test4/x_test4.max()
y_test4= y_test4/y_test4.max()
x_test4 = x_test4[:,:,:,np.newaxis]
y_test4 = y_test4[:,:,:,np.newaxis]

x_test5= x_test5/x_test5.max()
y_test5= y_test5/y_test5.max()
x_test5 = x_test5[:,:,:,np.newaxis]
y_test5 = y_test5[:,:,:,np.newaxis]

In [ ]:
!pip install -q keras

In [ ]:
!pip install git+https://github.com/qubvel/segmentation_models

In [ ]:
%env SM_FRAMEWORK=tf.keras

In [16]:
IMG_WIDTH = 256
IMG_HEIGHT = 256
IMG_CHANNELS = 1

In [ ]:
def AttentionBlock(x,shortcut,i_filters):
    g1 = Conv2D(i_filters,kernel_size = 1)(shortcut)
    g1 = BatchNormalization()(g1)
    x1 = Conv2D(i_filters,kernel_size = 1)(x)
    x1 = BatchNormalization()(x1)

    g1_x1 = Add()([g1,x1])
#    g1_x1 = concatenate([g1, x1], axis=3)
    psi = Activation('LeakyReLU')(g1_x1)
    psi = Conv2D(1,kernel_size = 1)(psi)
    psi = BatchNormalization()(psi)
    psi = Activation('sigmoid')(psi)
    x = Multiply()([x,psi])
    return x

def dice_coeff(y_true, y_pred):
    smooth = 1.
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    score = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    return score

In [17]:
def conv_block(x, filters, kernel_size=3, padding='same', strides=1):
    conv = bn_act(x)
    conv = Conv2D(filters, kernel_size, padding=padding, strides=strides)(conv)
    conv = Activation('relu')(conv)
    return conv

In [ ]:
def identity_block(x, f, filters):

  #retrieve filters (F1 = 64, F2 = 64, F3 = 256)

  F1, F2, F3 = filters
  x_shortcut = x

  # first layer

  x = Conv2D(filters = F1, kernel_size = (1, 1), strides = (1, 1), padding = 'valid')(x)
  x = BatchNormalization(axis = 1)(x)
  x = Activation('relu')(x)


  # second layer
  x = Conv2D(filters = F2, kernel_size = (f, f), strides = (1, 1), padding = 'same')(x)
  x = BatchNormalization(axis = 1)(x)
  x = Activation('relu')(x)

  # third layer

  x = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1, 1), padding = 'valid')(x)
  x = BatchNormalization(axis = 1)(x)
  x = Activation('relu')(x)

  # final step: Add shortcut value to F(x) and pass it through a relu activation
  x = Add()([x, x_shortcut])
  x= Activation('relu')(x)

  return x

In [ ]:
def convolutional_block(x, f, filters, s = 2):
  #retrieve filters
  F1, F2, F3 = filters

  x_shortcut = x

  # first layer

  x = Conv2D(filters = F1, kernel_size = (1, 1), strides = (1, 1), padding = 'valid'))x)
  x = Batchnormalization(axis = 1)(x)
  x = Activation('relu')(x)


  # second layer
  x = Conv2D(filters = F2, kernel_size = (f, f), strides = (1, 1), padding = 'same')(x)
  x = Batchnormalization(axis = 1)(x)
  x = Activation('relu')(x)

  # third layer

  x = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1, 1), padding = 'valid'))x)
  x = Batchnormalization(axis = 1)(x)

  ## shortcut path ####

  x_shortcut = Conv2D(filters = F3, kernel_size = (1, 1), strides = (s, s), padding = 'valid')(x_short_cut)
  x_shortcut = Batchnormalization(x_shortcut)

  ## add shortcut

  x = Add()([x, x_shortcut])
  x= Activation('relu')(x)

  return x

In [ ]:
def residual_block(x, filters = [64, 64, 256], kernel_size=[1,3,1], padding='same', strides=1):
    res = conv_block(x, filters[1], kernel_size[1], padding, strides)
    res = conv_block(res, filters[2], kernel_size[2], padding, 1)
    shortcut = Conv2D(filters[3], kernel_size[3], padding=padding, strides=strides)(x)
    shortcut = bn_act(shortcut, act=False)
    output = Add()([shortcut, res])
    return output

In [ ]:
def new_conv(x, filters= [64,64 ,256], kernel_size=[1,3,1], padding='same', strides=1):
    conv = conv_block(x, filters[1], kernel_size[1], padding, strides)
    conv = conv_block(conv, filters[2], kernel_size[2], padding, 1)
    shortcut = Conv2D(filters[3], kernel_size[3], padding=padding, strides=strides)(conv)
    shortcut = bn_act(shortcut, act=False)
    # shortcut path
    x_shortcut = Conv2D(filters[1] , kernel_size [1], strides=1 , padding = 'valid')(x)
    x_shortcut = BatchNormalization(x_shortcut)

    output = Add()([conv, x_shortcut])
    return output

In [ ]:
def stem(x, filters, kernel_size=7, padding='same', strides=1):
    x = ZeroPadding2D((3,3))(x) # 3x3 PADDING
    conv = Conv2D(filters, kernel_size, padding=padding, strides=strides)(x)
    x = BatchNormalization(axis = 1)(x)
    x= Activation('relu')(x)
    x = MaxPooling2D((3,3),strides=(2,2))(x)
    return x

In [ ]:
def bn_act(x, act=True):
    'batch normalization layer with an optinal activation layer'
    x = tf.keras.layers.BatchNormalization()(x)
    if act == True:
        x = tf.keras.layers.Activation('relu')(x)
    return x

In [ ]:
def upsample_concat_block(x, xskip):
    u = UpSampling2D((2,2))(x)
    c = Concatenate()([u, xskip])
    return c

In [ ]:
## Code for RESNET ATTENTION
def get_unet():

  inputs = tf.keras.layers.Input((IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))

  s = tf.keras.layers.Lambda(lambda x: x )(inputs)
  f = [16, 32, 64, 128, 256]
    #inputs = Input((img_h, img_w, 1))

    ## Encoder
  e0 = inputs
  e1 = stem(e0, f[0])
    # stage 2:

 # e2 = new_conv(e1, f[1])
  e2 = residual_block(e1, filters = [16, 16, 256], strides=1)
  e2 = residual_block(e2, f[1], strides=2)

 # Stage 3:
     # filter= [128,128,512],s=2
  e3 = new_conv(e2, f[2])
  e3 = residual_block(e2, f[2], strides=2)
  e3 = residual_block(e3, f[2], strides=2)
  e3 = residual_block(e3, f[2], strides=2)


 #Stage 4:
  e4 = new_conv(e3, f=3, filter= [256,256,1024],s=2)
  e4 = residual_block(e4, f[3], strides=2)
  e4 = residual_block(e4, f[3], strides=2)
  e4 = residual_block(e4, f[3], strides=2)
  e4 = residual_block(e4, f[3], strides=2)
  e4 = residual_block(e4, f[3], strides=2)

 # Stage 5:
  e5 = convolutional_block(e4, f=3, filter= [512,512,2048],s=2)
  e5 = residual_block(e5, f[4], strides=2)
  e5 = residual_block(e5, f[4], strides=2)

    ## Bridge
  b0 = conv_block(e5, f[4], strides=1)
  b1 = conv_block(b0, f[4], strides=1)
 #%
    # center
    #% upsampling %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
  up4 = UpSampling2D((2, 2))(b1)
  up4 = AttentionBlock(up4,e4,256)
  up4 = concatenate([e4, up4], axis=3)
  up4 = Conv2D(256, (2, 2),kernel_initializer= 'he_normal', padding='same')(up4)
  up4 = BatchNormalization()(up4)
  up4 = Activation('LeakyReLU')(up4)
  up4 = Conv2D(256, (2, 2),kernel_initializer= 'he_normal', padding='same')(up4)
  up4 = BatchNormalization()(up4)
  up4 = Activation('LeakyReLU')(up4)
#    up4 = Conv2D(64, (3, 3), padding='same')(up4)
#    up4 = BatchNormalization()(up4)
#    up4 = Activation('relu')(up4)
 #%
    # 16

  up3 = UpSampling2D((2, 2))(up4)
  up3 = AttentionBlock(up3,e3, 128)
  up3 = concatenate([e3, up3], axis=3)
  up3 = Conv2D(128, (2, 2),kernel_initializer= 'he_normal', padding='same')(up3)
  up3 = BatchNormalization()(up3)
  up3 = Activation('LeakyReLU')(up3)
  up3 = Conv2D(128, (2, 2),kernel_initializer= 'he_normal', padding='same')(up3)
  up3 = BatchNormalization()(up3)
  up3 = Activation('LeakyReLU')(up3)
#    up3 = Conv2D(32, (3, 3), padding='same')(up3)
#    up3 = BatchNormalization()(up3)
#    up3 = Activation('relu')(up3)
  #%# 32
  up2 = UpSampling2D((2, 2))(up3)
  up2 = AttentionBlock(up2,e2,64)
  up2 = concatenate([e2, up2], axis=3)
  up2 = Conv2D(64, (2, 2),kernel_initializer= 'he_normal', padding='same')(up2)
  up2 = BatchNormalization()(up2)
  up2 = Activation('LeakyReLU')(up2)
  up2 = Conv2D(64, (2, 2),kernel_initializer= 'he_normal', padding='same')(up2)
  up2 = BatchNormalization()(up2)
  up2 = Activation('LeakyReLU')(up2)
#    up2 = Conv2D(16, (3, 3), padding='same')(up2)
#    up2 = BatchNormalization()(up2)
#    up2 = Activation('relu')(up2)
    # 64
#%
  up1 = UpSampling2D((2, 2))(up2)
  up1 = AttentionBlock(up1,e1,32)
  up1 = concatenate([e1, up1], axis=3)
  up1 = Conv2D(32, (2, 2),kernel_initializer= 'he_normal', padding='same')(up1)
  up1 = BatchNormalization()(up1)
  up1 = Activation('LeakyReLU')(up1)

  dp2 = Dropout(0.5)(up1)
  up1 = Conv2D(32, (2, 2),kernel_initializer= 'he_normal', padding='same')(dp2)
  up1 = BatchNormalization()(up1)
  up1 = Activation('LeakyReLU')(up1)
#    up1 = Conv2D(8, (3, 3), padding='same')(up1)
#    up1 = BatchNormalization()(up1)
#    up1 = Activation('relu')(up1)
    # 128
#% output layer
#    classify = Conv2D(num_classes, (1, 1), activation='sigmoid')(up1)

#    dpup = Dropout(0.3)(up1)
  output_layer = Conv2D(filters=1, kernel_size=(1,1), activation='sigmoid')(up1)

  model = Model(inputs, output_layer)
  model.summary()
  return model

model = get_unet()

In [ ]:
model = ResNet50(input_shape = (224, 224, 3), classes = 3)

In [ ]:
model.compile(optimizer = 'adam', loss = 'categorial_crossentropy' metrics = ['accuracy'])

In [8]:
import numpy as np
x = np.load(r'D:\Anita_Mam\data256.npy')

In [9]:
x.shape

(5638, 256, 256)

In [ ]:
x_1 = x[1]
x_1.shape

In [7]:
import cv2
# x_1_read = cv2.imread(x_1)
cv2.imshow('Image', x_1)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
import numpy as np
y = np.load(r'D:\Anita_Mam\mask256.npy')

In [11]:
y.shape

(5638, 256, 256)

In [12]:
y_1 = y[1]
y_1.shape

(256, 256)

In [14]:
y_100 = y[100]
y_100.shape

import cv2
# x_1_read = cv2.imread(x_1)
cv2.imshow('Mask', y_100)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [15]:
x.shape == y.shape

True